**PySpark pour l'analyse défensive**

In [0]:
# 1. IMPORTATION DES FONCTIONS SPARK
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# 2. CHARGEMENT DES DONNÉES SOURCE (Indispensable pour l'indépendance du notebook)
schema = StructType([
    StructField("match_id", IntegerType(), True),
    StructField("home_team", StringType(), True),
    StructField("away_team", StringType(), True),
    StructField("home_goals", IntegerType(), True),
    StructField("away_goals", IntegerType(), True)
])

donnees_champions_league = [
    (1, "Paris SG", "Real Madrid", 3, 1), (2, "Bayern Munich", "Arsenal", 2, 2),
    (3, "FC Barcelone", "Monaco", 1, 2), (4, "Manchester City", "Juventus", 4, 0),
    (5, "Inter Milan", "Liverpool", 0, 2), (6, "Bayer Leverkusen", "AC Milan", 2, 1),
    (7, "Atletico Madrid", "RB Leipzig", 1, 0), (8, "Borussia Dortmund", "Sporting CP", 3, 2),
    (9, "Benfica", "Feyenoord", 1, 1), (10, "PSV Eindhoven", "Girona", 2, 0),
    (11, "Aston Villa", "Bologna", 2, 1), (12, "Celtic Glasgow", "Club Brugge", 0, 0),
    (13, "Stuttgart", "Lille OSC", 1, 2), (14, "Eintracht Frankfurt", "Lazio", 2, 3),
    (15, "Real Sociedad", "AS Roma", 1, 1), (16, "Shakhtar Donetsk", "Dinamo Zagreb", 0, 2),
    (17, "Salzburg", "Brest", 1, 3), (18, "Sparta Prague", "Young Boys", 2, 0),
    (19, "Sturm Graz", "Sporting CP", 0, 2), (20, "Crvena Zvezda", "Monaco", 1, 4),
    (21, "Real Madrid", "Borussia Dortmund", 5, 2), (22, "Paris SG", "PSV Eindhoven", 1, 1),
    (23, "Juventus", "VfB Stuttgart", 0, 1), (24, "Arsenal", "Shakhtar Donetsk", 1, 0)
]

df_matches = spark.createDataFrame(donnees_champions_league, schema)

# Re-génération du classement de base (total_conceded, matches_played)
df_home = df_matches.select(F.col("home_team").alias("team"), F.col("home_goals").alias("goals_scored"), F.col("away_goals").alias("goals_conceded"))
df_away = df_matches.select(F.col("away_team").alias("team"), F.col("away_goals").alias("goals_scored"), F.col("home_goals").alias("goals_conceded"))
df_final_ladder = df_home.union(df_away).groupBy("team").agg(
    F.count("goals_scored").alias("matches_played"),
    F.sum("goals_conceded").alias("total_conceded")
)

# 3. CALCUL DE L'ANALYSE DÉFENSIVE (Votre code d'origine)
df_defense = df_final_ladder.withColumn(
    "Defensive_Ratio",
    F.round(F.col("total_conceded") / F.col("matches_played"), 2)
)

# 4. TRI DE LA MEILLEURE À LA PLUS FRAGILE DÉFENSE (Ordre croissant)
df_defense_sorted = df_defense.orderBy(F.col("Defensive_Ratio").asc())

# 5. SÉLECTION ET RENOMMAGE DES COLONNES POUR L'AFFICHAGE EN ANGLAIS
df_defense_display = df_defense_sorted.select(
    F.col("team").alias("Team"),
    F.col("matches_played").alias("Matches_Played"),
    F.col("total_conceded").alias("Goals_Conceded"),
    F.col("Defensive_Ratio").alias("Goals_Conceded_Per_Match")
)

display(df_defense_display)



Databricks visualization. Run in Databricks to view.